In [ ]:
#0-1　基本設定
!pip install -q numpy==1.26.4 catboost==1.2.7 optuna lightgbm scikit-learn pyarrow

In [ ]:
#0-2　基本設定

import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
from catboost import CatBoostClassifier, Pool
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import pickle
import gc
import warnings
import time
import seaborn as sns

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
N_FOLDS = 5
N_OPTUNA_TRIALS = 30
OPTUNA_FOLDS = 3          # Optuna探索時のFold数（速度優先）
OPTUNA_SAMPLE_FRAC = 0.25 # Optuna探索時のデータサンプリング率

%matplotlib inline
%config Inlinebackend.figure_format='retina'
sns.set(style='whitegrid')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',100)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#0-3 データ読込
train=pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/加工データ/03FE2/train_FE2.zstd.parquet')
test =pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/加工データ/03FE2/test_FE2.zstd.parquet')

print(f"train: {train.shape}")
print(f"test:  {test.shape}")
print(f"TARGET分布:\n{train['TARGET'].value_counts(normalize=True)}")

train: (307511, 671)
test:  (48744, 670)
TARGET分布:
TARGET
0.0    0.919271
1.0    0.080729
Name: proportion, dtype: float64


In [ ]:
'''
0 読み込み（FE2パーケット）
1 下準備(ターゲット分離)
2 Optuna（LGBM）
  Optuna（CatBoost）
3 LGBM本番学習（最適パラメータ　StratifiedKFold）
  CatBoost本番学習（最適パラメータ　StratifiedKFold）
  アンサンブル（重み調整）
4 テスト予測、出力

以下の手順は異なるノートブック(アナライズ)で実施
(SHAP、寄与度分析、混同行列、閾値最適化など技術的な分析など)
5 寄与度
6 SHAP
'''

'\n0 読み込み（FE2パーケット）\n1 下準備(ターゲット分離)\n2 Optuna（LGBM）\n  Optuna（CatBoost）\n3 LGBM本番学習（最適パラメータ\u3000StratifiedKFold）\n  CatBoost本番学習（最適パラメータ\u3000StratifiedKFold）\n  アンサンブル（重み調整）\n4 テスト予測、出力\n5 寄与度\n6 SHAP\n'

 | モデル | Public LB (参考) |
 |--------|-----------------|
 | LGBM ベースライン | 0.77854 |
 | LGBM FE2 | 0.78711 |
 | CatBoost FE2 | 0.78924 |
 | Ensemble FE2 | 0.79083 |

In [ ]:
#1-1　下準備(ID・ターゲット分離)

TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

y = train[TARGET].values
test_ids = test[ID_COL].values

drop_cols = [TARGET, ID_COL] if ID_COL in train.columns else [TARGET]
X = train.drop(columns=drop_cols)
X_test = test.drop(columns=[c for c in drop_cols if c in test.columns])

# カラム順序を揃える
X_test = X_test[X.columns]
print(f"特徴量数: {X.shape[1]}")

# --- カテゴリ列の特定 ---
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"カテゴリ列数: {len(cat_cols)}")
if cat_cols:
    print(f"例: {cat_cols[:10]}")

特徴量数: 669
カテゴリ列数: 39
例: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


In [ ]:
#1-2　下準備(Optuna用サンプリング)
from sklearn.model_selection import train_test_split
X_sample, _, y_sample, _ = train_test_split(
    X, y, train_size=OPTUNA_SAMPLE_FRAC, stratify=y, random_state=SEED
)
print(f"Optuna探索用データ: {X_sample.shape} (元: {X.shape})")
print(f"  TARGET=1比率: {y_sample.mean():.4f} (元: {y.mean():.4f})")

print(f"X_sample: {X_sample.shape}, TARGET=1率: {y_sample.mean():.4f}")

Optuna探索用データ: (76877, 669) (元: (307511, 669))
  TARGET=1比率: 0.0807 (元: 0.0807)
X_sample: (76877, 669), TARGET=1率: 0.0807


In [ ]:
#2-1　ハイパーパラメーターチューニング(Optuna　LGBM)

# def objective_lgbm(trial):
#     """LightGBM用Optuna目的関数（サンプリングデータ使用）"""
#     params = {
#         "objective": "binary",
#         "metric": "auc",
#         "boosting_type": "gbdt",
#         "verbosity": -1,
#         "n_jobs": -1,
#         "seed": SEED,
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
#         "num_leaves": trial.suggest_int("num_leaves", 20, 150),
#         "max_depth": trial.suggest_int("max_depth", 3, 12),
#         "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
#         "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
#         "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
#         "subsample": trial.suggest_float("subsample", 0.5, 1.0),
#         "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
#         "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 15.0),
#         "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
#     }

#     skf = StratifiedKFold(n_splits=OPTUNA_FOLDS, shuffle=True, random_state=SEED)
#     auc_scores = []

#     for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_sample, y_sample)):
#         X_tr, X_va = X_sample.iloc[tr_idx], X_sample.iloc[va_idx]
#         y_tr, y_va = y_sample[tr_idx], y_sample[va_idx]

#         dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols, free_raw_data=False)
#         dval   = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols, free_raw_data=False)

#         model = lgb.train(
#             params,
#             dtrain,
#             num_boost_round=1500,  # Optuna探索用（本番は5000）
#             valid_sets=[dval],
#             callbacks=[
#                 lgb.early_stopping(stopping_rounds=100, verbose=False),
#                 lgb.log_evaluation(period=0),
#             ],
#         )
#         preds = model.predict(X_va, num_iteration=model.best_iteration)
#         auc_scores.append(roc_auc_score(y_va, preds))

#         del dtrain, dval, model
#         gc.collect()

#     return np.mean(auc_scores)


# # %%
# print("=" * 60)
# print("LightGBM Optuna 探索開始")
# print("=" * 60)

# study_lgbm = optuna.create_study(direction="maximize", study_name="lgbm_hc")
# study_lgbm.optimize(objective_lgbm, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

# print(f"\nベスト AUC: {study_lgbm.best_value:.5f}")
# print(f"ベストパラメータ:")
# for k, v in study_lgbm.best_params.items():
#     print(f"  {k}: {v}")


In [ ]:
'''
============================================================
LightGBM Optuna 探索開始
============================================================
Best trial: 26. Best value: 0.772907: 100%
 30/30 [3:17:18<00:00, 351.45s/it]

ベスト AUC: 0.77291
ベストパラメータ:
  learning_rate: 0.010034324967976509
  num_leaves: 135
  max_depth: 5
  min_child_samples: 89
  reg_alpha: 1.1088909181883837
  reg_lambda: 0.0002545303189986067
  colsample_bytree: 0.6536445553807908
  subsample: 0.7699138316650453
  subsample_freq: 7
  scale_pos_weight: 1.117960433760057
  min_split_gain: 0.8545296987304584
  '''

'\n============================================================\nLightGBM Optuna 探索開始\n============================================================\nBest\u2007trial:\u200726.\u2007Best\u2007value:\u20070.772907:\u2007100%\n\u200730/30\u2007[3:17:18<00:00,\u2007351.45s/it]\n\nベスト AUC: 0.77291\nベストパラメータ:\n  learning_rate: 0.010034324967976509\n  num_leaves: 135\n  max_depth: 5\n  min_child_samples: 89\n  reg_alpha: 1.1088909181883837\n  reg_lambda: 0.0002545303189986067\n  colsample_bytree: 0.6536445553807908\n  subsample: 0.7699138316650453\n  subsample_freq: 7\n  scale_pos_weight: 1.117960433760057\n  min_split_gain: 0.8545296987304584\n  '

In [ ]:
#2-2　ハイパーパラメーターチューニング(Optuna　CatBoost)

# # --- CatBoost用前処理 ---

#＊＊＊＊＊＊＊＊＊＊＊＊ここはコメントアウトしないこと＊＊＊＊＊＊＊＊＊＊＊＊
#モデル実行時に使用するため
def prepare_catboost_data(df, cat_columns):
    """CatBoostが受け付ける形にカテゴリ列を変換する。
    downcast済みデータの型衝突を回避するため、NumPy経由で処理。
    """
    df_out = df.copy()
    for col in cat_columns:
        arr = df_out[col].values.astype(str)
        arr = np.where(arr == "nan", "Unknown", arr)
        df_out[col] = arr
    return df_out

X_cat = prepare_catboost_data(X, cat_cols)
X_test_cat = prepare_catboost_data(X_test, cat_cols)

cat_indices = [X_cat.columns.get_loc(c) for c in cat_cols]
#＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊＊

# # Optuna用サンプル（CatBoost版）
# X_cat_sample = X_cat.loc[X_sample.index]
# y_cat_sample = y_sample

# # %%
# def objective_catboost(trial):
#     """CatBoost用Optuna目的関数"""
#     params = {
#         "loss_function": "Logloss",
#         "eval_metric": "AUC",
#         "task_type": "CPU",
#         "thread_count": -1,
#         "random_seed": SEED,
#         "verbose": 0,
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
#         "depth": trial.suggest_int("depth", 4, 10),
#         "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
#         "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
#         "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
#         "border_count": trial.suggest_int("border_count", 32, 255),
#         "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 15.0),
#         "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
#     }

#     skf = StratifiedKFold(n_splits=OPTUNA_FOLDS, shuffle=True, random_state=SEED)
#     auc_scores = []

#     for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_cat_sample, y_cat_sample)):
#         X_tr = X_cat_sample.iloc[tr_idx]
#         X_va = X_cat_sample.iloc[va_idx]
#         y_tr, y_va = y_cat_sample[tr_idx], y_cat_sample[va_idx]

#         pool_tr = Pool(X_tr, label=y_tr, cat_features=cat_indices)
#         pool_va = Pool(X_va, label=y_va, cat_features=cat_indices)

#         model = CatBoostClassifier(
#             iterations=1500,  # Optuna探索用（本番は3000）
#             early_stopping_rounds=100,
#             **params,
#         )
#         model.fit(pool_tr, eval_set=pool_va, verbose=0)

#         preds = model.predict_proba(pool_va)[:, 1]
#         auc_scores.append(roc_auc_score(y_va, preds))

#         del pool_tr, pool_va, model
#         gc.collect()

#     return np.mean(auc_scores)


# # %%
# print("=" * 60)
# print("CatBoost Optuna 探索開始")
# print("=" * 60)

# study_cb = optuna.create_study(direction="maximize", study_name="catboost_hc")
# study_cb.optimize(objective_catboost, n_trials=10, show_progress_bar=True)

# print(f"\nベスト AUC: {study_cb.best_value:.5f}")
# print(f"ベストパラメータ:")
# for k, v in study_cb.best_params.items():
#     print(f"  {k}: {v}")

【1回目】CatBoost Optuna 探索開始  
Best trial: 10. Best value: 0.773492:  37%
 11/30 [6:57:39<13:27:21, 2549.56s/it]  
進捗11/30でセッションが切れたたため、トライアル10で再走  
  
GPU枠が復活したら以下のコード書き換えで実行可能だが、制限化での環境も考慮し、CPUでの実装を行う。  
【CAT】  
・オプチュナの書き換え  
"task_type": "CPU",     → "GPU",  
・モデルの書き換え  
"thread_count": -1,     → "devices": "0",  

best_cb_params = {  
    "loss_function": "Logloss",  
    "eval_metric": "AUC",  
    "task_type": "GPU",  
    "thread_count": -1,  
    "random_seed": SEED,  
    "verbose": 0,  
    **study_cb.best_params,  
}

メモ  
オプチュナのみ他で実行し、結果をハードコードして用いることで解消できるが、今回は制限化での環境(守秘義務等で持出不可等)を想定してCPUで実行。  

以下、GPU使用可能リスト  
Kaggle: 週30時間、T4/P100  
Lightning AI: 月22時間のGPU無料枠。VSCode風  
Paperspace (by DigitalOcean): 無料GPUがあったが最近は枠が不安定  
Google Cloud 無料クレジット: 新規アカウントで$300分。GPUインスタンスを時間借りできる

In [ ]:
'''
============================================================
CatBoost Optuna 探索開始
============================================================
Best trial: 9. Best value: 0.770619: 100%
 10/10 [4:19:58<00:00, 1720.86s/it]

ベスト AUC: 0.77062
ベストパラメータ:
  learning_rate: 0.019602696816445685
  depth: 5
  l2_leaf_reg: 0.011913774630957736
  bagging_temperature: 0.8628119513308402
  random_strength: 7.841763317060064
  border_count: 175
  scale_pos_weight: 3.741754904098568
  min_data_in_leaf: 36
  '''

'\n============================================================\nCatBoost Optuna 探索開始\n============================================================\nBest\u2007trial:\u20079.\u2007Best\u2007value:\u20070.770619:\u2007100%\n\u200710/10\u2007[4:19:58<00:00,\u20071720.86s/it]\n\nベスト AUC: 0.77062\nベストパラメータ:\n  learning_rate: 0.019602696816445685\n  depth: 5\n  l2_leaf_reg: 0.011913774630957736\n  bagging_temperature: 0.8628119513308402\n  random_strength: 7.841763317060064\n  border_count: 175\n  scale_pos_weight: 3.741754904098568\n  min_data_in_leaf: 36\n  '

In [ ]:
#3-1　LGBM本番学習（最適パラメータ　StratifiedKFold）

# best_lgbm_params = {
#     "objective": "binary",
#     "metric": "auc",
#     "boosting_type": "gbdt",
#     "verbosity": -1,
#     "n_jobs": -1,
#     "seed": SEED,
#     "learning_rate": 0.010034324967976509,
#     "num_leaves": 135,
#     "max_depth": 5,
#     "min_child_samples": 89,
#     "reg_alpha": 1.1088909181883837,
#     "reg_lambda": 0.0002545303189986067,
#     "colsample_bytree": 0.6536445553807908,
#     "subsample": 0.7699138316650453,
#     "subsample_freq": 7,
#     "scale_pos_weight": 1.117960433760057,
#     "min_split_gain": 0.8545296987304584,
# }

# skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# oof_lgbm = np.zeros(len(X))
# preds_lgbm = np.zeros(len(X_test))
# lgbm_models = []
# lgbm_importances = []

# print("=" * 60)
# print("LightGBM 本番学習")
# print("=" * 60)

# for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
#     t0 = time.time()
#     X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
#     y_tr, y_va = y[tr_idx], y[va_idx]

#     dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols, free_raw_data=False)
#     dval   = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols, free_raw_data=False)

#     model = lgb.train(
#         best_lgbm_params,
#         dtrain,
#         num_boost_round=5000,
#         valid_sets=[dval],
#         callbacks=[
#             lgb.early_stopping(stopping_rounds=200, verbose=False),
#             lgb.log_evaluation(period=500),
#         ],
#     )

#     oof_lgbm[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
#     preds_lgbm += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS

#     # 特徴量重要度
#     imp = pd.DataFrame({
#         "feature": X.columns,
#         "importance": model.feature_importance(importance_type="gain"),
#         "fold": fold_idx,
#     })
#     lgbm_importances.append(imp)
#     lgbm_models.append(model)

#     auc = roc_auc_score(y_va, oof_lgbm[va_idx])
#     elapsed = time.time() - t0
#     print(f"  Fold {fold_idx}: AUC={auc:.5f}  best_iter={model.best_iteration}  ({elapsed:.0f}s)")

#     del dtrain, dval
#     gc.collect()

# oof_auc_lgbm = roc_auc_score(y, oof_lgbm)
# print(f"\n>>> LightGBM OOF AUC: {oof_auc_lgbm:.5f}")

In [ ]:
'''
============================================================
LightGBM 本番学習
============================================================
[500]	valid_0's auc: 0.772817
[1000]	valid_0's auc: 0.781486
[1500]	valid_0's auc: 0.784879
[2000]	valid_0's auc: 0.78694
[2500]	valid_0's auc: 0.788133
[3000]	valid_0's auc: 0.788689
  Fold 0: AUC=0.78877  best_iter=2830  (1245s)
[500]	valid_0's auc: 0.78235
[1000]	valid_0's auc: 0.792178
[1500]	valid_0's auc: 0.795999
[2000]	valid_0's auc: 0.79806
[2500]	valid_0's auc: 0.799171
[3000]	valid_0's auc: 0.799675
[3500]	valid_0's auc: 0.799949
  Fold 1: AUC=0.79998  best_iter=3530  (1450s)
[500]	valid_0's auc: 0.772397
[1000]	valid_0's auc: 0.782225
[1500]	valid_0's auc: 0.7857
[2000]	valid_0's auc: 0.787412
[2500]	valid_0's auc: 0.788204
[3000]	valid_0's auc: 0.788525
[3500]	valid_0's auc: 0.788885
  Fold 2: AUC=0.78895  best_iter=3528  (1447s)
[500]	valid_0's auc: 0.779652
[1000]	valid_0's auc: 0.788029
[1500]	valid_0's auc: 0.791567
[2000]	valid_0's auc: 0.793381
[2500]	valid_0's auc: 0.794473
[3000]	valid_0's auc: 0.794753
[3500]	valid_0's auc: 0.795254
  Fold 3: AUC=0.79533  best_iter=3394  (1402s)
[500]	valid_0's auc: 0.771868
[1000]	valid_0's auc: 0.781799
[1500]	valid_0's auc: 0.785904
[2000]	valid_0's auc: 0.788048
[2500]	valid_0's auc: 0.789092
[3000]	valid_0's auc: 0.790039
[3500]	valid_0's auc: 0.79034
[4000]	valid_0's auc: 0.790641
  Fold 4: AUC=0.79066  best_iter=4003  (1591s)

>>> LightGBM OOF AUC: 0.79271
'''

"\n============================================================\nLightGBM 本番学習\n============================================================\n[500]\tvalid_0's auc: 0.772817\n[1000]\tvalid_0's auc: 0.781486\n[1500]\tvalid_0's auc: 0.784879\n[2000]\tvalid_0's auc: 0.78694\n[2500]\tvalid_0's auc: 0.788133\n[3000]\tvalid_0's auc: 0.788689\n  Fold 0: AUC=0.78877  best_iter=2830  (1245s)\n[500]\tvalid_0's auc: 0.78235\n[1000]\tvalid_0's auc: 0.792178\n[1500]\tvalid_0's auc: 0.795999\n[2000]\tvalid_0's auc: 0.79806\n[2500]\tvalid_0's auc: 0.799171\n[3000]\tvalid_0's auc: 0.799675\n[3500]\tvalid_0's auc: 0.799949\n  Fold 1: AUC=0.79998  best_iter=3530  (1450s)\n[500]\tvalid_0's auc: 0.772397\n[1000]\tvalid_0's auc: 0.782225\n[1500]\tvalid_0's auc: 0.7857\n[2000]\tvalid_0's auc: 0.787412\n[2500]\tvalid_0's auc: 0.788204\n[3000]\tvalid_0's auc: 0.788525\n[3500]\tvalid_0's auc: 0.788885\n  Fold 2: AUC=0.78895  best_iter=3528  (1447s)\n[500]\tvalid_0's auc: 0.779652\n[1000]\tvalid_0's auc: 0.78802

In [ ]:
#3-2　LGBM結果保存（後続のCAT実行時のセッション切れ対策）

# from google.colab import drive
# drive.mount('/content/drive')

# import os
# SAVE_DIR = "/content/drive/MyDrive/home_credit_models"
# os.makedirs(SAVE_DIR, exist_ok=True)

# # モデル保存
# for i, mdl in enumerate(lgbm_models):
#     with open(os.path.join(SAVE_DIR, f"lgbm_fold{i}.pkl"), "wb") as f:
#         pickle.dump(mdl, f)

# # OOF・テスト予測保存
# pd.DataFrame({
#     "SK_ID_CURR": train["SK_ID_CURR"].values,
#     "oof_lgbm": oof_lgbm,
#     "TARGET": y,
# }).to_parquet(os.path.join(SAVE_DIR, "oof_lgbm.parquet"), index=False)

# pd.DataFrame({
#     "SK_ID_CURR": test_ids,
#     "pred_lgbm": preds_lgbm,
# }).to_parquet(os.path.join(SAVE_DIR, "pred_lgbm.parquet"), index=False)

# # 特徴量重要度
# imp_df = pd.concat(lgbm_importances)
# imp_df.to_parquet(os.path.join(SAVE_DIR, "lgbm_importance.parquet"), index=False)

# print(f"LGBM中間保存完了: {os.listdir(SAVE_DIR)}")

In [ ]:
#3-3　LGBM結果の復元

import os

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/home_credit_models"

oof_lgbm_df = pd.read_parquet(os.path.join(SAVE_DIR, "oof_lgbm.parquet"))
oof_lgbm = oof_lgbm_df["oof_lgbm"].values
y = oof_lgbm_df["TARGET"].values

pred_lgbm_df = pd.read_parquet(os.path.join(SAVE_DIR, "pred_lgbm.parquet"))
preds_lgbm = pred_lgbm_df["pred_lgbm"].values
test_ids = pred_lgbm_df["SK_ID_CURR"].values

oof_auc_lgbm = roc_auc_score(y, oof_lgbm)
print(f"LGBM OOF AUC (復元): {oof_auc_lgbm:.5f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LGBM OOF AUC (復元): 0.79271


In [ ]:
#3-4　CatBoost本番学習（最適パラメータ　StratifiedKFold）

# best_cb_params = {
#     "loss_function": "Logloss",
#     "eval_metric": "AUC",
#     "task_type": "CPU",
#     "thread_count": -1,
#     "random_seed": SEED,
#     "verbose": 0,
#     "learning_rate": 0.019602696816445685,
#     "depth": 5,
#     "l2_leaf_reg": 0.011913774630957736,
#     "bagging_temperature": 0.8628119513308402,
#     "random_strength": 7.841763317060064,
#     "border_count": 175,
#     "scale_pos_weight": 3.741754904098568,
#     "min_data_in_leaf": 36,
# }

# skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# from google.colab import drive
# drive.mount('/content/drive')

# SAVE_DIR = "/content/drive/MyDrive/home_credit_models"

# oof_cb = np.zeros(len(X_cat))
# preds_cb = np.zeros(len(X_test_cat))

# print("=" * 60)
# print("CatBoost 本番学習")
# print("=" * 60)

# for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_cat, y)):
#     t0 = time.time()
#     X_tr = X_cat.iloc[tr_idx]
#     X_va = X_cat.iloc[va_idx]
#     y_tr, y_va = y[tr_idx], y[va_idx]

#     pool_tr = Pool(X_tr, label=y_tr, cat_features=cat_indices)
#     pool_va = Pool(X_va, label=y_va, cat_features=cat_indices)

#     model = CatBoostClassifier(
#         iterations=3000,
#         early_stopping_rounds=200,
#         **best_cb_params,
#     )
#     model.fit(pool_tr, eval_set=pool_va, verbose=500)

#     oof_cb[va_idx] = model.predict_proba(pool_va)[:, 1]
#     preds_cb += model.predict_proba(Pool(X_test_cat, cat_features=cat_indices))[:, 1] / N_FOLDS

#     auc = roc_auc_score(y_va, oof_cb[va_idx])
#     elapsed = time.time() - t0
#     print(f"  Fold {fold_idx}: AUC={auc:.5f}  best_iter={model.best_iteration_}  ({elapsed:.0f}s)")

#     # 寄与度取得
#     imp = pd.DataFrame({
#         "feature": X_cat.columns,
#         "importance": model.get_feature_importance(),
#         "fold": fold_idx,
#     })
#     imp.to_parquet(os.path.join(SAVE_DIR, f"catboost_imp_fold{fold_idx}.parquet"), index=False)

#     # --- Fold単位で即保存 & メモリ解放 ---
#     model.save_model(os.path.join(SAVE_DIR, f"catboost_fold{fold_idx}.cbm"))
#     del pool_tr, pool_va, model
#     gc.collect()
#     print(f"  Fold {fold_idx} 保存・メモリ解放完了")

# oof_auc_cb = roc_auc_score(y, oof_cb)
# print(f"\n>>> CatBoost OOF AUC: {oof_auc_cb:.5f}")

In [ ]:
'''
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
============================================================
CatBoost 本番学習
============================================================
0:	test: 0.6531547	best: 0.6531547 (0)	total: 3.89s	remaining: 3h 14m 34s
500:	test: 0.7558089	best: 0.7558089 (500)	total: 18m 10s	remaining: 1h 30m 41s
1000:	test: 0.7754745	best: 0.7754745 (1000)	total: 36m 23s	remaining: 1h 12m 41s
1500:	test: 0.7825291	best: 0.7825291 (1500)	total: 54m 37s	remaining: 54m 33s
2000:	test: 0.7850514	best: 0.7850621 (1993)	total: 1h 12m 48s	remaining: 36m 21s
2500:	test: 0.7859486	best: 0.7859636 (2494)	total: 1h 30m 48s	remaining: 18m 7s
2999:	test: 0.7865383	best: 0.7865564 (2995)	total: 1h 48m 35s	remaining: 0us

bestTest = 0.7865563668
bestIteration = 2995

Shrink model to first 2996 iterations.
  Fold 0: AUC=0.78656  best_iter=2995  (6546s)
  Fold 0 保存・メモリ解放完了
0:	test: 0.6863230	best: 0.6863230 (0)	total: 1.88s	remaining: 1h 33m 44s
500:	test: 0.7641381	best: 0.7641381 (500)	total: 17m 26s	remaining: 1h 27m 1s
1000:	test: 0.7850115	best: 0.7850115 (1000)	total: 35m 34s	remaining: 1h 11m 2s
1500:	test: 0.7919078	best: 0.7919078 (1500)	total: 53m 52s	remaining: 53m 47s
2000:	test: 0.7941190	best: 0.7941190 (2000)	total: 1h 12m 6s	remaining: 36m
2500:	test: 0.7951268	best: 0.7951543 (2497)	total: 1h 30m 11s	remaining: 17m 59s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7951952327
bestIteration = 2529

Shrink model to first 2530 iterations.
  Fold 1: AUC=0.79520  best_iter=2529  (5925s)
  Fold 1 保存・メモリ解放完了
0:	test: 0.6709908	best: 0.6709908 (0)	total: 1.97s	remaining: 1h 38m 35s
500:	test: 0.7555013	best: 0.7555013 (500)	total: 17m 20s	remaining: 1h 26m 29s
1000:	test: 0.7759076	best: 0.7759076 (1000)	total: 35m 17s	remaining: 1h 10m 29s
1500:	test: 0.7825179	best: 0.7825179 (1500)	total: 53m 38s	remaining: 53m 34s
2000:	test: 0.7850125	best: 0.7850247 (1996)	total: 1h 11m 55s	remaining: 35m 54s
2500:	test: 0.7863508	best: 0.7863681 (2498)	total: 1h 29m 55s	remaining: 17m 56s
2999:	test: 0.7869286	best: 0.7869759 (2977)	total: 1h 47m 48s	remaining: 0us

bestTest = 0.7869758811
bestIteration = 2977

Shrink model to first 2978 iterations.
  Fold 2: AUC=0.78698  best_iter=2977  (6492s)
  Fold 2 保存・メモリ解放完了
0:	test: 0.6849956	best: 0.6849956 (0)	total: 1.98s	remaining: 1h 38m 49s
500:	test: 0.7653276	best: 0.7653276 (500)	total: 17m 43s	remaining: 1h 28m 25s
1000:	test: 0.7838046	best: 0.7838046 (1000)	total: 35m 50s	remaining: 1h 11m 33s
1500:	test: 0.7899962	best: 0.7900005 (1499)	total: 54m 20s	remaining: 54m 15s
2000:	test: 0.7924681	best: 0.7924681 (2000)	total: 1h 12m 45s	remaining: 36m 19s
2500:	test: 0.7937448	best: 0.7937801 (2451)	total: 1h 31m 5s	remaining: 18m 10s
2999:	test: 0.7945890	best: 0.7945977 (2998)	total: 1h 49m 19s	remaining: 0us

bestTest = 0.7945976898
bestIteration = 2998

Shrink model to first 2999 iterations.
  Fold 3: AUC=0.79460  best_iter=2998  (6582s)
  Fold 3 保存・メモリ解放完了
0:	test: 0.6763276	best: 0.6763276 (0)	total: 1.91s	remaining: 1h 35m 24s
500:	test: 0.7533246	best: 0.7533246 (500)	total: 17m 46s	remaining: 1h 28m 37s
1000:	test: 0.7743827	best: 0.7743827 (1000)	total: 36m 10s	remaining: 1h 12m 14s
1500:	test: 0.7821486	best: 0.7821486 (1500)	total: 54m 26s	remaining: 54m 21s
2000:	test: 0.7843063	best: 0.7843118 (1979)	total: 1h 12m 38s	remaining: 36m 15s
2500:	test: 0.7855889	best: 0.7855889 (2500)	total: 1h 30m 47s	remaining: 18m 6s
2999:	test: 0.7864717	best: 0.7864724 (2996)	total: 1h 48m 55s	remaining: 0us

bestTest = 0.786472351
bestIteration = 2996

Shrink model to first 2997 iterations.
  Fold 4: AUC=0.78647  best_iter=2996  (6557s)
  Fold 4 保存・メモリ解放完了

>>> CatBoost OOF AUC: 0.78992
'''

'\nDrive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).\n============================================================\nCatBoost 本番学習\n============================================================\n0:\ttest: 0.6531547\tbest: 0.6531547 (0)\ttotal: 3.89s\tremaining: 3h 14m 34s\n500:\ttest: 0.7558089\tbest: 0.7558089 (500)\ttotal: 18m 10s\tremaining: 1h 30m 41s\n1000:\ttest: 0.7754745\tbest: 0.7754745 (1000)\ttotal: 36m 23s\tremaining: 1h 12m 41s\n1500:\ttest: 0.7825291\tbest: 0.7825291 (1500)\ttotal: 54m 37s\tremaining: 54m 33s\n2000:\ttest: 0.7850514\tbest: 0.7850621 (1993)\ttotal: 1h 12m 48s\tremaining: 36m 21s\n2500:\ttest: 0.7859486\tbest: 0.7859636 (2494)\ttotal: 1h 30m 48s\tremaining: 18m 7s\n2999:\ttest: 0.7865383\tbest: 0.7865564 (2995)\ttotal: 1h 48m 35s\tremaining: 0us\n\nbestTest = 0.7865563668\nbestIteration = 2995\n\nShrink model to first 2996 iterations.\n  Fold 0: AUC=0.78656  best_iter=2995  (6546s)

In [ ]:
#3-5　CAT結果保存
# #　再試走に5～7時間程度かかる見込みのため、放置後のセッション切れを考慮して保存

# # OOF予測保存
# pd.DataFrame({
#     "SK_ID_CURR": train["SK_ID_CURR"].values,
#     "oof_catboost": oof_cb,
#     "TARGET": y,
# }).to_parquet(os.path.join(SAVE_DIR, "oof_catboost.parquet"), index=False)

# # テスト予測保存
# pd.DataFrame({
#     "SK_ID_CURR": test_ids,
#     "pred_catboost": preds_cb,
# }).to_parquet(os.path.join(SAVE_DIR, "pred_catboost.parquet"), index=False)

# print(f"CatBoost結果保存完了: {os.listdir(SAVE_DIR)}")
# print(f"CatBoost OOF AUC: {oof_auc_cb:.5f}")

In [ ]:
#3-6　CAT結果の復元

import os
from catboost import CatBoostClassifier

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/home_credit_models"
N_FOLDS = 5

# --- OOF予測の復元 ---
oof_cb = pd.read_parquet(os.path.join(SAVE_DIR, "oof_catboost.parquet"))["oof_catboost"].values

# --- テスト予測の復元 ---
pred_cb = pd.read_parquet(os.path.join(SAVE_DIR, "pred_catboost.parquet"))
preds_cb = pred_cb["pred_catboost"].values

# --- モデルの復元 ---
models_cat = []
for i in range(N_FOLDS):
    model = CatBoostClassifier()
    model.load_model(os.path.join(SAVE_DIR, f"catboost_fold{i}.cbm"))
    models_cat.append(model)

# --- 寄与度の復元 ---
catboost_importances = []
for i in range(N_FOLDS):
    imp = pd.read_parquet(os.path.join(SAVE_DIR, f"catboost_imp_fold{i}.parquet"))
    catboost_importances.append(imp)

# --- 確認 ---
from sklearn.metrics import roc_auc_score
oof_auc_cb = roc_auc_score(y, oof_cb)
print(f"CatBoost OOF AUC (復元): {oof_auc_cb:.5f}")
print(f"モデル数: {len(models_cat)}")
print(f"テスト予測shape: {preds_cb.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CatBoost OOF AUC (復元): 0.78992
モデル数: 5
テスト予測shape: (48744,)


In [ ]:
#3-7　アンサンブル（重み調整）

def find_best_weight(oof_a, oof_b, y_true, steps=101):
    """OOFスコアで最適なブレンド重みをグリッドサーチする。"""
    best_w, best_auc = 0.0, 0.0
    results = []
    for w in np.linspace(0, 1, steps):
        blended = w * oof_a + (1 - w) * oof_b
        auc = roc_auc_score(y_true, blended)
        results.append((w, auc))
        if auc > best_auc:
            best_w, best_auc = w, auc
    return best_w, best_auc, results

w_lgbm, ensemble_auc, weight_results = find_best_weight(oof_lgbm, oof_cb, y)

print("=" * 60)
print("アンサンブル結果")
print("=" * 60)
print(f"最適重み: LGBM={w_lgbm:.2f}, CatBoost={1 - w_lgbm:.2f}")
print(f"Ensemble OOF AUC: {ensemble_auc:.5f}")
print(f"  (LGBM単体: {oof_auc_lgbm:.5f})")
print(f"  (CatBoost単体: {oof_auc_cb:.5f})")

アンサンブル結果
最適重み: LGBM=0.82, CatBoost=0.18
Ensemble OOF AUC: 0.79343
  (LGBM単体: 0.79271)
  (CatBoost単体: 0.78992)


In [ ]:
#4-1　テスト予測、出力

# --- テスト予測のブレンド ---
preds_ensemble = w_lgbm * preds_lgbm + (1 - w_lgbm) * preds_cb

# --- 提出ファイル作成 ---
sub = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: preds_ensemble,
})
sub.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv 作成完了: {sub.shape}")
print(sub.head())


submission.csv 作成完了: (48744, 2)
   SK_ID_CURR    TARGET
0      100001  0.080040
1      100005  0.152661
2      100013  0.053442
3      100028  0.035876
4      100038  0.214700


Ensemble OOF AUC: 0.79343  
プライベートスコア：0.79234  （アンサンブルAUCから-0.00109）  
パブリックスコア：0.79556  　（アンサンブルAUCから+0.00213)   
  
モデルAUCと近いスコアであることから過学習に陥っておらず、実装後も安定した精度が保てると考えられる。

In [ ]:
#5-1　モデル・予測値の保存(アンサンブル)
#LGBM,CATは実施済のためアンサンブルのみ実施

# --- アンサンブル結果の保存 ---
SAVE_DIR = "/content/drive/MyDrive/home_credit_models"

# OOF（アンサンブル込み）
oof_all = pd.DataFrame({
    "SK_ID_CURR": train["SK_ID_CURR"].values,
    "oof_lgbm": oof_lgbm,
    "oof_catboost": oof_cb,
    "oof_ensemble": w_lgbm * oof_lgbm + (1 - w_lgbm) * oof_cb,
    "TARGET": y,
})
oof_all.to_parquet(os.path.join(SAVE_DIR, "oof_all.parquet"), index=False)

# テスト予測（アンサンブル込み）
test_all = pd.DataFrame({
    "SK_ID_CURR": test_ids,
    "pred_lgbm": preds_lgbm,
    "pred_catboost": preds_cb,
    "pred_ensemble": preds_ensemble,
})
test_all.to_parquet(os.path.join(SAVE_DIR, "test_all.parquet"), index=False)

# submission
sub = pd.DataFrame({"SK_ID_CURR": test_ids, "TARGET": preds_ensemble})
sub.to_csv(os.path.join(SAVE_DIR, "submission.csv"), index=False)

print(f"保存完了: {os.listdir(SAVE_DIR)}")

保存完了: ['lgbm_fold0.pkl', 'lgbm_fold1.pkl', 'lgbm_fold2.pkl', 'lgbm_fold3.pkl', 'lgbm_fold4.pkl', 'oof_lgbm.parquet', 'pred_lgbm.parquet', 'lgbm_importance.parquet', 'catboost_imp_fold0.parquet', 'catboost_fold0.cbm', 'catboost_imp_fold1.parquet', 'catboost_fold1.cbm', 'catboost_imp_fold2.parquet', 'catboost_fold2.cbm', 'catboost_imp_fold3.parquet', 'catboost_fold3.cbm', 'catboost_imp_fold4.parquet', 'catboost_fold4.cbm', 'oof_catboost.parquet', 'pred_catboost.parquet', 'oof_all.parquet', 'test_all.parquet', 'submission.csv']



 まとめ
 | モデル | Public LB (参考) |
 |--------|-----------------|
 | LGBM ベースライン | 0.77854 |
 | LGBM FE2 | 0.78711 |
 | CatBoost FE2 | 0.78924 |
 | Ensemble FE2 | 0.79083 |


| 項目 | 変数名 | スコア |
|:---|:---|:---|
| LGBM OOF AUC | `oof_auc_lgbm` | 0.79271 |
| CatBoost OOF AUC | `oof_auc_cb` | 0.78992 |
| Ensemble OOF AUC | `ensemble_auc` | 0.79343 |
| Private Score | --- | 0.79234 |
| Public Score | --- | 0.79556 |

パラメーターチューニングにより、LGBMは差分+0.00560と大きく改善された一方でCATは+0.00068と微増となった。  
この点についてはオプチュナ実行時のCPUの限界からトライアルを10で実施したためだと思われる。  
今回は制限化を想定したため、CPUで実装を行ったが、GPUでLGBM、CATのトライアルを50にした状態でスコアがどれほど異なるのかも検証したい。

なお、プライベートスコア0.79234は当時のリーダーボードで1423位と同じスコアであった。  
(1423/7180　上位20％以内)